# TurboLLM on Kaggle — one-click (dual T4)

Run a full **TurboLLM** server with its web GUI on Kaggle's free **2× Tesla T4**, using a
**prebuilt CUDA engine** (no 40-minute compile). You bring the model — download one inside
the GUI, or attach a Kaggle model.

### Before you press ▸▸ Run All
1. **Settings → Accelerator → GPU T4 × 2**
2. **Settings → Internet → On**
3. **Add Input → Datasets → search `turboquant-cuda-t4` → Add** (the prebuilt engine)

Then **Run All**. At the end, open the printed **GUI URL** and paste the **Token**.

## 1 · Preflight — fail fast if the session isn't set up
Checks the 3 requirements above so you never wait on a misconfigured run.

In [ ]:
import glob, subprocess, time, urllib.request
smi = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
gpus = [l for l in smi.stdout.strip().splitlines() if l.strip()]
print("\n".join(gpus) or smi.stderr)
assert len(gpus) >= 2, f"Need 2 GPUs — Settings → Accelerator → GPU T4 ×2 (found {len(gpus)})"
# Kaggle auto-extracts an uploaded .tar.gz on dataset creation, so the engine can show up EITHER
# as the tarball itself or as the already-extracted bin/llama-server + its .so files — accept
# both. Mount depth varies by Kaggle account/notebook type (observed /kaggle/input/<slug>/… AND
# /kaggle/input/datasets/<owner>/<slug>/…), so use a recursive glob instead of a fixed depth.
eng = (glob.glob("/kaggle/input/**/turboquant-cuda-t4*.tar.gz", recursive=True)
       or glob.glob("/kaggle/input/**/bin/llama-server", recursive=True))
assert eng, "Engine bundle missing — Add Input → Datasets → 'turboquant-cuda-t4' → Add"
print("engine bundle:", eng[0])
# A freshly-started Kaggle container's DNS resolver can be flaky for the first few seconds
# (live-verified: "Temporary failure in name resolution" on the very first outbound request of a
# brand-new session, on more than one otherwise-healthy session) — retry before treating it as
# "Internet is off", so a real cold-start blip doesn't fail the whole run.
last_err = None
for attempt in range(5):
    try:
        urllib.request.urlopen("https://github.com", timeout=8)
        last_err = None
        break
    except Exception as e:
        last_err = e
        print(f"  internet check attempt {attempt + 1}/5 failed ({e}) — retrying…")
        time.sleep(5)
if last_err is not None:
    raise SystemExit(f"Internet is OFF — Settings → Internet → On ({last_err})")
print("\nPreflight OK — 2 GPUs, engine attached, internet on.")

## 2 · Configure
Extract the engine to `/tmp` (keeps the 3.8 GB engine off the 19.5 GB `/kaggle/working`
quota, so that space stays free for models you download in the GUI), and ship no baked-in model.

In [ ]:
import os
os.environ["TURBOLLM_ENGINE_ROOT"] = "/tmp/turboquant"
os.environ["TURBOLLM_SKIP_MODEL"]  = "1"
# Branch to clone in Section 3. The dual-GPU placement fix this notebook was built to
# exercise shipped in v1.11.4, so this tracks main; point it at a branch to test one.
os.environ["TURBOLLM_BRANCH"]      = "main"
print("engine root:", os.environ["TURBOLLM_ENGINE_ROOT"],
      "| skip baked-in model:", os.environ["TURBOLLM_SKIP_MODEL"],
      "| branch:", os.environ["TURBOLLM_BRANCH"])

## 3 · Get TurboLLM
Clones the public repo (once) at `TURBOLLM_BRANCH` (default `main`). Re-running is a no-op.

In [ ]:
![ -d /kaggle/working/TurboLLM/.git ] || git clone --depth 1 -b "$TURBOLLM_BRANCH" https://github.com/mohitsoni48/TurboLLM.git /kaggle/working/TurboLLM
!cd /kaggle/working/TurboLLM && git log --oneline -1

## 4 · Setup (prebuilt engine — ~5 min the first time)
Installs Node, the daemon deps, and the web UI, and unpacks the prebuilt CUDA engine from the
attached dataset. No compile, no model download. Idempotent — safe to re-run.

In [ ]:
!bash /kaggle/working/TurboLLM/deploy/kaggle/setup.sh

## 5 · Launch — GUI over a public tunnel
Starts the daemon, activates the CUDA engine, registers the model directories, and opens a
public `*.trycloudflare.com` tunnel to the GUI.

In [ ]:
!bash /kaggle/working/TurboLLM/deploy/kaggle/serve.sh start

In [ ]:
import re, pathlib, time
log = pathlib.Path("/kaggle/working/turbollm-daemon.log")
url = tok = None
for _ in range(30):
    t = log.read_text(errors="ignore") if log.exists() else ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", t)
    k = re.search(r"Token:\s*(\S+)", t)
    if m: url = m.group(0)
    if k: tok = k.group(1)
    if url and tok: break
    time.sleep(2)
print("\n" + "=" * 60)
print("  OPEN YOUR TURBOLLM GUI")
print("=" * 60)
print("  URL  :", url or "(not found — see Section 5 output above)")
print("  Token:", tok or "(not found — see Section 5 output above)")
print("=" * 60)

## Done — using the GUI

Open the **URL** above and paste the **Token**. Then:

- **Load a model** — either **Models → Download** a GGUF from Hugging Face (saved to
  `/kaggle/working/models`), **or** attach a Kaggle model via **Add Input → Models** (served
  from `/kaggle/input`). Both directories are already registered.
- **Auto-tune** the model, then **Chat**. Both T4s are used automatically (layer-split across
  the two cards); for a model that fits on one card, auto-tune keeps it on a single GPU for speed.

The tunnel stays up while this notebook session is running. If it drops, re-run **Section 5**
for a fresh URL + token.

> **Note:** Kaggle can't pin the *number* of GPUs from notebook metadata — if Preflight says it
> found fewer than 2, set **Settings → Accelerator → GPU T4 × 2** and Run All again.

In [ ]:
# Keep the tunnel alive.
#
# A Kaggle *batch* run (`kaggle kernels push`) ends the moment the last cell finishes, which
# would tear the tunnel down seconds after it came up. Interactive sessions stay up on their
# own, so this only blocks under Batch - `Run All` in the editor still completes normally.
#
# TURBOLLM KEEP-ALIVE
import os, re, pathlib, time

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") != "Batch":
    print("Interactive session - tunnel stays up while this notebook is running.")
else:
    log = pathlib.Path("/kaggle/working/turbollm-daemon.log")
    HOURS = 9
    for minute in range(HOURS * 60):
        if minute % 10 == 0:
            t = log.read_text(errors="ignore") if log.exists() else ""
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", t)
            k = re.search(r"Token:\s*(\S+)", t)
            print(f"[keep-alive {minute // 60}h{minute % 60:02d}m] "
                  f"URL: {m.group(0) if m else '?'}  Token: {k.group(1) if k else '?'}",
                  flush=True)
        time.sleep(60)
